<a href="https://colab.research.google.com/github/A-ros1076/BUS118s/blob/Dev/Exercise_1_Prompt_Chaining_for_a_Customer_Support_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 1: Prompt Chaining for a Customer Support AI

**Goal:** Build a multi-step prompt chain where each step's output feeds the next, simulating a complete customer service resolution flow.

| Step | Task | Input | Output |
|------|------|-------|--------|
| 1 | Collect message + classify issue | Customer's typed message | Issue type + severity |
| 2 | Gather missing info | Step 1 classification | Targeted follow-up question + customer answer |
| 3 | Propose a solution | Steps 1 + 2 + customer answer | Recommended resolution |
| 4 | Escalation check | Step 1 severity + Step 3 solution | Resolve or escalate to human agent |

**Tool:** Google Colab + Gemini API (`gemini-2.5-flash`)

In [1]:
!pip install google-generativeai -q

In [2]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel('gemini-2.5-flash')

def ask_gemini(prompt: str) -> str:
    """
    Sends a prompt to Gemini and returns the text response.

    Args:
        prompt: The full prompt string to send.

    Returns:
        Gemini's response as a stripped string.
    """
    try:
        response = model.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        print(f"An error occurred: {e}")
        return "Request failed."


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## Prompt Chain — All Steps

The full 4-step chain runs in one cell. Each step's output is passed directly into the next prompt.

**Iteration note:** The initial Step 3 prompt said only *"draft a helpful response."* Gemini returned vague language with no timeline and ignored the customer's urgency. After adding explicit constraints (acknowledge urgency, give a realistic timeline, avoid hollow promises), the output quality improved significantly — see the rules block in Step 3 below.

In [3]:
# ============================================================
# STEP 1 — Collect customer message and classify the issue
# ============================================================
print("Hello! I'm your customer support AI.")
print("Please describe the issue you're experiencing:")
customer_message = input("You: ")

print("\n--- Step 1: Classifying your issue... ---")
step1_prompt = f"""
You are a customer support triage specialist.
Read the customer message below and classify it.

Respond in EXACTLY this format — two lines, no extra text:
Issue Type: [one of: Billing, Technical, Shipping, Account, General]
Severity: [one of: Low, Medium, High, Critical]

Customer message:
\"{customer_message}\"
"""
step1_output = ask_gemini(step1_prompt)
print(step1_output)

# ============================================================
# STEP 2 — Use Step 1 output to ask a targeted follow-up
# ============================================================
print("\n--- Step 2: Follow-up question ---")
step2_prompt = f"""
You are a customer support agent.
A customer submitted a support request classified as:
{step1_output}

Their original message:
\"{customer_message}\"

Ask the customer exactly ONE clarifying question to get the most critical missing information.
Rules:
- One question only.
- Polite and professional tone.
- Maximum 2 sentences.
- Do not repeat information the customer already provided.
- No greeting or sign-off, just the question.
"""
step2_output = ask_gemini(step2_prompt)
print(f"AI: {step2_output}")
customer_answer = input("You: ")

# ============================================================
# STEP 3 — Use Steps 1 + 2 to propose a solution
# ============================================================
print("\n--- Step 3: Generating solution... ---")
step3_prompt = f"""
You are a senior customer support specialist.
Use all context below to draft a resolution response for the customer.

Issue Classification (Step 1):
{step1_output}

Clarifying question asked (Step 2):
{step2_output}

Customer's answer:
\"{customer_answer}\"

Original message:
\"{customer_message}\"

Rules:
- Professional and empathetic tone.
- Acknowledge urgency if severity is High or Critical.
- Give a specific, actionable resolution with a realistic timeline.
- Do not make promises you cannot guarantee (avoid 'we will fix this immediately').
- Maximum 5 sentences.
"""
step3_output = ask_gemini(step3_prompt)
print(f"AI: {step3_output}")

# ============================================================
# STEP 4 — Use Step 1 severity + Step 3 solution to decide escalation
# ============================================================
print("\n--- Step 4: Escalation check... ---")
step4_prompt = f"""
You are a customer support team lead reviewing a proposed resolution.

Issue severity (Step 1):
{step1_output}

Proposed resolution (Step 3):
{step3_output}

Escalation rules — escalate if ANY of these apply:
- Severity is High or Critical
- Issue involves both billing AND account access problems at the same time
- Resolution requires actions a front-line agent cannot perform alone
- Customer has an urgent time constraint not fully addressed by the resolution

Start your response with exactly one word: RESOLVED or ESCALATE
Then give the reason in 1-2 sentences. No greeting or sign-off.
"""
step4_output = ask_gemini(step4_prompt)
print(step4_output)

# ============================================================
# FINAL STATUS
# ============================================================
print("\n" + "=" * 60)
if step4_output.upper().startswith("ESCALATE"):
    print("FINAL STATUS: ESCALATED TO HUMAN AGENT")
    print("AI: Your case has been escalated to a specialist who will follow up shortly. We apologize for the inconvenience.")
else:
    print("FINAL STATUS: RESOLVED BY AI AGENT")
    print("AI: Your issue has been resolved. Please let us know if there is anything else we can help with!")
print("=" * 60)

Hello! I'm your customer support AI.
Please describe the issue you're experiencing:
You: My laptop wont turn on 

--- Step 1: Classifying your issue... ---
Issue Type: Technical
Severity: High

--- Step 2: Follow-up question ---
AI: Are there any lights, sounds, or fan activity when you try to power it on?
You: no

--- Step 3: Generating solution... ---
AI: I understand how critical it is to have your laptop operational, and I apologize for the inconvenience this high-priority issue is causing. Given there are no lights or sounds when attempting to power on, it indicates a core power issue requiring immediate attention. Please perform a hard reset by disconnecting all peripherals and the AC adapter, then hold the power button for 30 seconds before reconnecting and attempting to power on again. If this does not resolve the issue, we will need to schedule a hardware diagnostic. A technician can be available to further assist within the next 24-48 hours.

--- Step 4: Escalation check... -